# FarmTech Solutions — Ir Além: Otimização com Algoritmo Genético
**Fase 7 - Projeto Integrado**

Integrantes do grupo:
- Caroline Coelho Mendes — RM570370
- Leandro Paiva — RM572159
- Lucas Viana de Lima — RM571835

---
### Objetivo do notebook
A ideia desse teste é resolver o problema de alocação de recursos da fazenda usando algoritmo genético (baseado no problema da mochila).
A gente tem 12 talhões e precisa escolher quais vão receber água e adubo para dar o máximo de sacas colhidas, mas sem ultrapassar o limite de água (700 m³) e nem o orçamento de insumos (25 mil reais).

### 1. Criando e salvando a tabela de talhões
Aqui a gente cria a tabela com os dados dos talhões e salva em csv pra poder reutilizar sem mudar os números.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

# semente fixa pros números não mudarem a cada execução
np.random.seed(42)

dados = {
    'talhao_id': [f'Talhao_{i+1}' for i in range(12)],
    'cultura': ['Milho', 'Soja', 'Milho', 'Feijao', 'Soja', 'Milho', 'Soja', 'Algodao', 'Milho', 'Soja', 'Milho', 'Feijao'],
    'consumo_agua_m3': [120, 95, 140, 70, 85, 130, 90, 160, 110, 100, 135, 75],
    'custo_insumos_mil_reais': [4.2, 3.1, 5.0, 2.5, 3.0, 4.8, 3.2, 6.0, 4.0, 3.5, 4.9, 2.7],
    'produtividade_sacas': [180, 140, 210, 90, 135, 195, 145, 230, 170, 150, 205, 100]
}

df_talhoes = pd.DataFrame(dados)
df_talhoes.to_csv('insumos_otimizacao.csv', index=False)
print('Arquivo salvo com sucesso!')
df_talhoes

### 2. Função do Algoritmo Genético
Criamos o algoritmo com duas opções de seleção: roleta e torneio. A função calcula o fitness somando as sacas, mas zera o resultado se estourar o limite de água ou o orçamento.

In [ ]:
def rodar_algoritmo_genetico(df, limite_agua=700, limite_custo=25.0, tam_pop=40, geracoes=60, mutacao=0.05, tipo_selecao='roleta'):
    total_talhoes = len(df)
    agua = df['consumo_agua_m3'].values
    custos = df['custo_insumos_mil_reais'].values
    sacas = df['produtividade_sacas'].values

    # calcula a pontuacao de cada combinacao
    def avaliar(individuo):
        gasto_agua = np.sum(individuo * agua)
        gasto_custo = np.sum(individuo * custos)
        # se passar do limite, penaliza
        if gasto_agua > limite_agua or gasto_custo > limite_custo:
            return 1e-4
        return np.sum(individuo * sacas)

    # comeca com combinacoes aleatorias de 0 e 1
    populacao = np.random.randint(0, 2, size=(tam_pop, total_talhoes))
    historico = []
    inicio = time.time()

    for g in range(geracoes):
        notas = np.array([avaliar(ind) for ind in populacao])
        historico.append(np.max(notas))

        # escolha dos pais
        if tipo_selecao == 'torneio':
            pais = []
            for _ in range(tam_pop):
                i1, i2 = np.random.choice(tam_pop, 2, replace=False)
                pais.append(populacao[i1] if notas[i1] > notas[i2] else populacao[i2])
            populacao = np.array(pais)
        else: # roleta simples
            soma = np.sum(notas)
            chances = notas / soma if soma > 0 else np.ones(tam_pop) / tam_pop
            idx = np.random.choice(tam_pop, size=tam_pop, p=chances)
            populacao = populacao[idx]

        # cruzamento entre pares
        filhos = []
        for i in range(0, tam_pop, 2):
            p1, p2 = populacao[i], populacao[(i + 1) % tam_pop]
            corte = np.random.randint(1, total_talhoes)
            f1 = np.concatenate([p1[:corte], p2[corte:]])
            f2 = np.concatenate([p2[:corte], p1[corte:]])
            filhos.extend([f1, f2])
        populacao = np.array(filhos[:tam_pop])

        # mutacao sorteada
        sorteio = np.random.rand(*populacao.shape) < mutacao
        populacao = np.bitwise_xor(populacao, sorteio.astype(int))

    tempo = time.time() - inicio
    notas_finais = np.array([avaliar(ind) for ind in populacao])
    melhor = populacao[np.argmax(notas_finais)]

    return {
        'metodo': tipo_selecao,
        'melhor': melhor,
        'max_sacas': np.max(notas_finais),
        'agua_gasta': np.sum(melhor * agua),
        'custo_gasto': np.sum(melhor * custos),
        'tempo': tempo,
        'historico': historico
    }

# rodando os testes com os dois metodos
resultado_roleta = rodar_algoritmo_genetico(df_talhoes, tipo_selecao='roleta')
resultado_torneio = rodar_algoritmo_genetico(df_talhoes, tipo_selecao='torneio')

print(f"Roleta -> Sacas: {resultado_roleta['max_sacas']} | Tempo: {resultado_roleta['tempo']:.4f}s")
print(f"Torneio -> Sacas: {resultado_torneio['max_sacas']} | Tempo: {resultado_torneio['tempo']:.4f}s")

### 3. Comparando o resultado dos dois métodos no gráfico
O gráfico abaixo mostra a evolução da quantidade de sacas a cada geração.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(resultado_roleta['historico'], label='Selecao Roleta', color='blue')
plt.plot(resultado_torneio['historico'], label='Selecao Torneio', color='green', linestyle='--')
plt.title('Comparativo de Evolucao das Geracoes')
plt.xlabel('Geracao')
plt.ylabel('Total de Sacas')
plt.legend()
plt.grid(True)
plt.savefig('grafico_convergencia.png')
plt.show()

### 4. Talhões escolhidos no melhor resultado

In [ ]:
escolhidos = [i for i, v in enumerate(resultado_torneio['melhor']) if v == 1]
print('Talhoes que devem receber manejo nesta safra:')
df_talhoes.iloc[escolhidos]